In [1]:
import os
import glob
import pandas as pd

def inspect_nids_datasets(folder_path):
    file_list = glob.glob(os.path.join(folder_path, "*.csv"))
    if not file_list:
        print(f"경로에 CSV 파일이 없습니다: {folder_path}")
        return

    print(f"=== 총 {len(file_list)}개의 파일 검사 시작 ===\n")
    
    all_features_set = set()
    first_file_features = None
    is_all_same = True

    for idx, file in enumerate(file_list):
        file_name = os.path.basename(file)
        print(f"[{idx + 1}] 파일명: {file_name}")
        
        # 첫 행만 읽어서 피처 이름과 개수 파악 (속도 최적화)
        df_header = pd.read_csv(file, nrows=0)
        features = list(df_header.columns)
        num_features = len(features)
        
        print(f"  - 총 피처 개수: {num_features}개")
        print(f"  - 피처 리스트: {features} ...")
        
        # 파일 간 피처가 일치하는지 비교 구문
        if idx == 0:
            first_file_features = set(features)
        else:
            if set(features) != first_file_features:
                is_all_same = False
                print(f"주의: 첫 번째 파일과 피처 구성이 다릅니다!")
        
        # 정답 레이블 분포 확인을 위해 'Label' 열만 전체 로드
        try:
            df_label = pd.read_csv(file, usecols=['Label'])
            # 문자열 공백 제거 후 카운트
            label_counts = df_label['Label'].str.strip().value_counts()
            print("  - 레이블(정답) 분포:")
            for lbl, count in label_counts.items():
                print(f"    * {lbl}: {count:,}개")
        except ValueError:
            print("에러: 이 파일에는 'Label' 열이 존재하지 않습니다.")
        
        print("-" * 50)

    print("\n=== 최종 분석 결과 ===")
    if is_all_same:
        print("모든 CSV 파일의 피처 이름과 개수가 정확히 일치합니다.")
    else:
        print("파일마다 피처 구성이 다릅니다! 통합 전 전처리(피처 맞춤)가 필요합니다.")

# 실행부 (본인의 경로에 맞게 수정하여 사용하세요)
if __name__ == "__main__":
    folder_path = r"C:\ids2018_data"
    inspect_nids_datasets(folder_path)

=== 총 10개의 파일 검사 시작 ===

[1] 파일명: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  - 총 피처 개수: 80개
  - 피처 리스트: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Siz

In [2]:
import os
import glob
import numpy as np
import pandas as pd

def clean_and_unify_labels(df):
    if 'Label' not in df.columns:
        return df
    df = df[df['Label'].astype(str).str.strip() != 'Label']
    df['Label'] = df['Label'].astype(str).str.strip()
    
    label_mapping = {
        'Benign': 'Benign', 'Bot': 'Bot',
        'DoS attacks-Hulk': 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest': 'DoS attacks-SlowHTTPTest',
        'DoS attacks-GoldenEye': 'DoS attacks-GoldenEye', 'DoS attacks-Slowloris': 'DoS attacks-Slowloris',
        'Brute Force -Web': 'Brute Force - Web', 'Brute Force - Web': 'Brute Force - Web',
        'Brute Force -XSS': 'Brute Force - XSS', 'Brute Force - XSS': 'Brute Force - XSS',
        'SQL Injection': 'SQL Injection', 'Infilteration': 'Infiltration', 'Infiltration': 'Infiltration',
        'FTP-BruteForce': 'FTP-BruteForce', 'SSH-Bruteforce': 'SSH-BruteForce', 'SSH-BruteForce': 'SSH-BruteForce',
        'DDOS attack-HOIC': 'DDoS attacks-HOIC', 'DDoS attacks-LOIC-HTTP': 'DDoS attacks-LOIC-HTTP',
        'DDOS attack-LOIC-UDP': 'DDoS attacks-LOIC-UDP'
    }
    df['Label'] = df['Label'].map(label_mapping).fillna(df['Label'])
    return df

def merge_and_smart_clean(folder_path, output_filename="nids_advanced_cleaned.csv"):
    file_list = glob.glob(os.path.join(folder_path, "*.csv"))
    if not file_list:
        print(f"경로에 CSV 파일이 없습니다: {folder_path}")
        return None
    
    print(f"총 {len(file_list)}개의 파일 발견. 통합 및 스마트 정제 시작...")
    
    df_list = []
    for file in file_list:
        print(f" -> 로드 중: {os.path.basename(file)}")
        df = pd.read_csv(file, low_memory=False)
        df = clean_and_unify_labels(df)
        df_list.append(df)
        
    # 공통 피처 기준 통합
    merged_df = pd.concat(df_list, axis=0, join='inner', ignore_index=True)
    del df_list
    
    # 수치형 열 정의
    numeric_cols = merged_df.select_dtypes(include=[np.number]).columns
    
    # 무한대 값(inf)을 NaN으로 치환
    merged_df[numeric_cols] = merged_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    
    # 통합 데이터 기준 '통째로 비어있거나 모든 값이 동일(분산=0)'한 열 필터링 후 삭제
    cols_to_drop = [col for col in numeric_cols if merged_df[col].isna().all() or merged_df[col].nunique() <= 1]
    if cols_to_drop:
        print(f"   통합 후 통째로 비어있거나 무의미한 {len(cols_to_drop)}개의 열을 삭제합니다.")
        print(f"   삭제된 열 목록: {cols_to_drop}")
        merged_df = merged_df.drop(columns=cols_to_drop)
        # 수치형 열 목록 갱신
        numeric_cols = [c for c in numeric_cols if c not in cols_to_drop]

    print("[효과적 결측치 처리] 레이블(정답)별 그룹 중앙값 적용 중...")
    
    # 레이블 그룹별 중앙값으로 결측치 채우기
    # 정상 데이터의 빈칸은 정상 중앙값으로, 공격 데이터의 빈칸은 공격 중앙값으로 정밀하게 채움
    for col in numeric_cols:
        # 각 레이블 그룹별 중앙값 계산 후 대입
        merged_df[col] = merged_df.groupby('Label')[col].transform(lambda x: x.fillna(x.median()))
        
        # 혹시 특정 공격 그룹 전체가 해당 피처가 비어있어 미처 못 채워진 경우, 최종적으로 전체 중앙값으로 보완
        if merged_df[col].isna().any():
            overall_median = merged_df[col].median()
            merged_df[col] = merged_df[col].fillna(overall_median if not pd.isna(overall_median) else 0.0)

    print(f"   스마트 정제 완료! 최종 데이터 크기: {merged_df.shape}")
    
    output_path = os.path.join(folder_path, output_filename)
    print(f"최종 저장 중 -> {output_path}")
    merged_df.to_csv(output_path, index=False)
    print("작업 완료.")
    return merged_df

if __name__ == "__main__":
    target_folder = r"C:\ids2018_data" 
    final_df = merge_and_smart_clean(target_folder)

총 10개의 파일 발견. 통합 및 스마트 정제 시작...
 -> 로드 중: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].astype(str).str.strip()
C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].map(label_mapping).fillna(df['Label'])


 -> 로드 중: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].astype(str).str.strip()
C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].map(label_mapping).fillna(df['Label'])


 -> 로드 중: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
 -> 로드 중: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].astype(str).str.strip()
C:\Users\USER\AppData\Local\Temp\ipykernel_26920\2298688633.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].map(label_mapping).fillna(df['Label'])


💡 [효과적 결측치 처리] 레이블(정답)별 그룹 중앙값 적용 중...
   스마트 정제 완료! 최종 데이터 크기: (16232943, 80)
최종 저장 중 -> C:\ids2018_data\nids_advanced_cleaned.csv
작업 완료.


In [3]:
import os
import numpy as np
import pandas as pd

def absolute_clean_and_verify(file_path):
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print("==================================================")
    print("무한값 및 결측치 완벽 제거 작업")
    print("==================================================")
    
    # 데이터 로드
    df = pd.read_csv(file_path, low_memory=False)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # 남아있는 모든 inf / -inf 값을 다시 한 번 확실하게 NaN으로 치환
    print("문자열 유입 또는 누락된 무한값(inf)을 NaN으로 강제 변환 중...")
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    
    # 2차 보완 기법 (Fallback) 적용
    print("클래스별 데이터 전멸로 누락된 결측치 2차 보완 중 (전체 중앙값 대체)...")
    for col in numeric_cols:
        # 그룹별 중앙값으로 안 채워진 빈칸이 있다면 해당 열의 전체 중앙값으로 채움
        if df[col].isna().any():
            overall_median = df[col].median()
            # 만약 열 전체가 NaN이라 대피값이 없으면 0.0으로 처리
            backup_value = overall_median if not pd.isna(overall_median) else 0.0
            df[col] = df[col].fillna(backup_value)
            
    # 최종 무결성 검증 (오류 확인용)
    print("\n==================================================")
    print("최종 정제 상태 재검증 결과")
    print("==================================================")
    
    final_nans = df.isna().sum().sum()
    final_infs = np.isinf(df[numeric_cols]).sum().sum()
    
    if final_nans == 0:
        print(" 결측치(NaN) 검증: 0개 (완벽히 제거됨)")
    else:
        print(f" 결측치(NaN) 에러: 여전히 {final_nans:,}개의 결측치가 존재합니다.")
        
    if final_infs == 0:
        print(" 무한값(inf) 검증: 0개 (완벽히 제거됨)")
    else:
        print(f" 무한값(inf) 에러: 여전히 {final_infs:,}개의 무한값이 존재합니다.")
        
    # 깨끗해진 데이터를 원본 파일에 덮어쓰기 저장
    if final_nans == 0 and final_infs == 0:
        print(f"\n정제 완료! 깨끗해진 데이터를 저장합니다 -> {file_path}")
        df.to_csv(file_path, index=False)
        print("모든 결함 데이터가 성공적으로 박멸되었습니다!")
    else:
        print("\n아직 데이터에 문제가 남아있어 저장하지 않았습니다. 로직을 재점검해야 합니다.")
    print("==================================================")

if __name__ == "__main__":
    # 분석 타깃 파일 경로
    merged_file_path = r"C:\ids2018_data\nids_advanced_cleaned.csv"
    
    absolute_clean_and_verify(merged_file_path)

무한값 및 결측치 완벽 제거 작업
문자열 유입 또는 누락된 무한값(inf)을 NaN으로 강제 변환 중...
클래스별 데이터 전멸로 누락된 결측치 2차 보완 중 (전체 중앙값 대체)...

최종 정제 상태 재검증 결과
 결측치(NaN) 검증: 0개 (완벽히 제거됨)
 무한값(inf) 검증: 0개 (완벽히 제거됨)

정제 완료! 깨끗해진 데이터를 저장합니다 -> C:\ids2018_data\nids_advanced_cleaned.csv
모든 결함 데이터가 성공적으로 박멸되었습니다!


In [4]:
import os
import numpy as np
import pandas as pd

def analyze_merged_dataset(file_path):
    if not os.path.exists(file_path):
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return

    print("==================================================")
    print(f"NIDS 통합 데이터셋 분석 시작: {os.path.basename(file_path)}")
    print("==================================================")
    
    # 1데이터 로드 (분석을 위해 상위 일부 및 전체 구조 파악)
    # 전체를 다 읽으면 시간이 걸릴 수 있으므로, 상태 확인용으로 로드
    df = pd.read_csv(file_path, low_memory=False)
    
    # 데이터셋 기본 크기 및 구조 출력
    print(f"\n[1] 데이터셋 기본 정보")
    print(f" - 전체 행(데이터) 수: {df.shape[0]:,}개")
    print(f" - 전체 열(피처) 수: {df.shape[1]}개")
    
    # head() 출력 (상위 5개 행)
    print(f"\n[2] 데이터셋 상위 5개 행 (head) 출력")
    # 모든 열이 잘 보이도록 판다스 출력 옵션 설정
    pd.set_option('display.max_columns', None)
    print(df.head())
    
    # 결측치(NaN) 존재 여부 검사
    print(f"\n[3] 결측치(NaN) 정제 결과 검증")
    nan_counts = df.isna().sum()
    total_nans = nan_counts.sum()
    if total_nans == 0:
        print("검증 완료: 데이터셋 내에 결측치(NaN)가 단 하나도 없습니다! (정상 처리됨)")
    else:
        print(f" 경고: 아직 처리되지 않은 결측치가 {total_nans:,}개 존재합니다.")
        print(nan_counts[nan_counts > 0])
        
    # 무한값(inf) 존재 여부 검사
    print(f"\n[4] 무한값(inf) 정제 결과 검증")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum().sum()
    if inf_counts == 0:
        print("검증 완료: 데이터셋 내에 무한대(inf, -inf) 값이 단 하나도 없습니다! (정상 처리됨)")
    else:
        print(f" 경고: 무한대(inf) 값이 {inf_counts:,}개 검출되었습니다. 추가 정제가 필요합니다.")

    # 최종 레이블(정답) 분포 및 오타 검증
    print(f"\n[5] 최종 레이블(정답) 클래스 분포 (오타/공백 검증)")
    if 'Label' in df.columns:
        label_counts = df['Label'].value_counts()
        for lbl, count in label_counts.items():
            print(f" - {lbl:<25}: {count:,}개")
        
        # 'Label' 문자열 정크가 남아있는지 재확인
        if 'Label' in label_counts.index:
            print("에러: 'Label'이라는 이름의 정크 행이 아직 남아있습니다!")
    else:
        print("에러: 데이터셋에 'Label' 열이 존재하지 않습니다.")
        
    print("\n==================================================")
    print("데이터셋 구조 및 정제 상태 검증 완료.")
    print("==================================================")

if __name__ == "__main__":
    # 병합된 파일이 저장된 경로를 정확히 입력해주세요.
    merged_file_path = r"C:\ids2018_data\nids_advanced_cleaned.csv"
    
    analyze_merged_dataset(merged_file_path)

NIDS 통합 데이터셋 분석 시작: nids_advanced_cleaned.csv

[1] 데이터셋 기본 정보
 - 전체 행(데이터) 수: 16,232,943개
 - 전체 열(피처) 수: 80개

[2] 데이터셋 상위 5개 행 (head) 출력
   Dst Port  Protocol            Timestamp  Flow Duration  Tot Fwd Pkts  \
0       443         6  02/03/2018 08:47:38         141385             9   
1     49684         6  02/03/2018 08:47:38            281             2   
2       443         6  02/03/2018 08:47:40         279824            11   
3       443         6  02/03/2018 08:47:40            132             2   
4       443         6  02/03/2018 08:47:41         274016             9   

   Tot Bwd Pkts  TotLen Fwd Pkts  TotLen Bwd Pkts  Fwd Pkt Len Max  \
0             7            553.0           3773.0            202.0   
1             1             38.0              0.0             38.0   
2            15           1086.0          10527.0            385.0   
3             0              0.0              0.0              0.0   
4            13           1285.0           6141.0            5